# Pixels to Predictions: CS-GY 6953 (Deep Learning) Kaggle Competition
## MODEL 9
### Author: Mariia Onokhina

---
### Experiment Results
- Model: SmolVLM-500M-Instruct (LoRA fine-tuned, 5-fold CV ensemble)
- Prompt: lecture_180 (same as Model 7, with metadata + truncated lecture)
- Training: 5-fold cross-validation (each model trained on ~80% of data)
- LoRA: r=8, alpha=16, dropout=0.05, target_modules=[q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj]
- Trainable params: ~3.9M (within constraint)
- Objective: Causal LM (predict answer index token)
- Epochs trained: 2 per fold
- Image size: 512
- Scoring: Index-only log-likelihood (mean aggregation)
- Ensemble: Averaged logits across 5 folds
- Best validation accuracy: ~0.5455
- Public Kaggle score: 0.54728
- Decision: Rejected - significantly worse than Model 7 (0.72233). CV ensemble reduced performance due to weaker per-fold models and undertraining.

---
**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

## Installation of Libraries 

Before installing any libraries, I create a new Conda environment and add it to Jupyter Notebook to ensure I start from a clean slate and that the code is reproducible.

**Run the following code in a terminal if you'd like to start fresh with a new environment:**
```bash
conda create -n pixels-to-predictions python=3.10
conda activate pixels-to-predictions
conda install -c conda-forge notebook
conda install -c conda-forge ipykernel
python -m ipykernel install --user --name pixels-to-predictions --display-name "Pixels-to-predictions DL"
```

IMPORTANT: Manually change the Kernel in Jupyter Notebook in VS Code or Jupyter Lab to "Pixels-to-predictions DL".

In [1]:
# Uncomment this cell to install the necessary Python packages.
import sys
print("Python:", sys.executable)
!{sys.executable} -m pip install -q transformers==4.57.6 peft==0.18.1 kaggle matplotlib scikit-learn pandas numpy ipywidgets jupyterlab_widgets bitsandbytes accelerate datasets pillow --quiet

Python: /home/devvingduo/miniforge3/envs/pixels-to-predictions/bin/python


---
## Imports & Configuration

In [2]:
# Imports & Configuration
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import ast
from transformers import AutoProcessor, AutoModelForVision2Seq

# For LoRa fine-tuning
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm   # For progress bar
from itertools import combinations
import gc
from collections import Counter

import pickle
from sklearn.model_selection import KFold
from PIL import Image

In [26]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [27]:
# This code makes sure it uses only 1 GPU of my choice
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
# Paths 
DATA_DIR = Path("../data")
IMAGE_ROOT = DATA_DIR / "images"

# Model
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings
IMG_SIZE = 512

# This run will be entirely with mean scoring
SCORE_MODE = "mean"

NUM_FOLDS = 5

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


---
### Load and Preprocess Data

Download data from https://www.kaggle.com/competitions/pixels-to-predictions/data via Kaggle CLI.

For this, you need a Legacy API key which you can create here: https://www.kaggle.com/settings.

When you create a new key, it will download a ```kaggle.json```. 

In your terminal, run:
```bash
mkdir -p ~/.kaggle
mv <your_downloads_folder> ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json
```

Verify that it worked by running: 
```bash
ls -la ~/.kaggle
```

I have to add it manually because I'm working via SSH into my Linux server machine.

In [30]:
# Uncomment this cell to download the data in a .zip file
#!kaggle competitions download -c pixels-to-predictions

In [31]:
# Uncomment this cell to unzip the data into "data" folder
#!unzip pixels-to-predictions.zip -d data

In [5]:
# Load CSVs
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

Inspect the data. 

In [6]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3109 entries, 0 to 3108
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           3109 non-null   object
 1   image_path   3109 non-null   object
 2   question     3109 non-null   object
 3   choices      3109 non-null   object
 4   num_choices  3109 non-null   int64 
 5   answer       3109 non-null   int64 
 6   hint         2385 non-null   object
 7   lecture      2669 non-null   object
 8   solution     2580 non-null   object
 9   task         3109 non-null   object
 10  grade        3109 non-null   object
 11  subject      3109 non-null   object
 12  topic        3109 non-null   object
 13  category     3109 non-null   object
 14  skill        3109 non-null   object
dtypes: int64(2), object(13)
memory usage: 364.5+ KB


In [7]:
# Function to parse the choices column using ast module (Abstract Syntax Tree)
def parse_choices(x):
    # If x is already a list, return it
    if isinstance(x, list):
        return x
    # If x is a string, parse it using ast.literal_eval
    return ast.literal_eval(x)

The ```choices``` column is a JSON string, so we parse it using the function above.

In [8]:
for df in [train_df, val_df, test_df]:
    df["choices_list"] = df["choices"].apply(parse_choices)

In [9]:
def load_image(path):
    image = Image.open(path).convert("RGB")
    image = image.resize((IMG_SIZE, IMG_SIZE))
    return image

---
### Create K-Folds

In [10]:
kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=42)

folds = []
for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    folds.append((train_idx, val_idx))

print("Total folds:", len(folds))

Total folds: 5


---
### Prompt

Keep `build_prompt_model2_imagecareful` which is a variation of Model 2's prompt. 

We want to see whether including lecture field in the prompt improves the score. It's very long so we truncate it.

In [11]:
def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x

In [12]:
def truncate_words(x, max_words=180):
    x = clean_text(x)
    if not x:
        return ""
    words = x.split()
    return " ".join(words[:max_words])

In [13]:
PROMPT_STYLE = "lecture_180"


def build_prompt(row):
    choices = row["choices_list"]

    choice_lines = "\n".join(
        [f"{i}: {choice}" for i, choice in enumerate(choices)]
    )

    prompt = "<image>\n"
    prompt += "Answer the science multiple-choice question.\n"
    prompt += "Look carefully at the image when it is relevant.\n"
    prompt += "Use the provided lecture and hint only if helpful.\n"
    prompt += "Return only the number of the correct choice.\n\n"

    grade = clean_text(row.get("grade", ""))
    subject = clean_text(row.get("subject", ""))
    topic = clean_text(row.get("topic", ""))
    category = clean_text(row.get("category", ""))
    skill = clean_text(row.get("skill", ""))

    if grade:
        prompt += f"Grade: {grade}\n"
    if subject:
        prompt += f"Subject: {subject}\n"
    if topic:
        prompt += f"Topic: {topic}\n"
    if category:
        prompt += f"Category: {category}\n"
    if skill:
        prompt += f"Skill: {skill}\n"

    lecture = clean_text(row.get("lecture", ""))

    if PROMPT_STYLE == "no_lecture":
        lecture = ""
    elif PROMPT_STYLE == "lecture_100":
        lecture = truncate_words(lecture, max_words=100)
    elif PROMPT_STYLE == "lecture_180":
        lecture = truncate_words(lecture, max_words=180)
    elif PROMPT_STYLE == "lecture_250":
        lecture = truncate_words(lecture, max_words=250)
    else:
        raise ValueError(f"Unknown PROMPT_STYLE: {PROMPT_STYLE}")

    if lecture:
        prompt += f"\nLecture:\n{lecture}\n"

    hint = clean_text(row.get("hint", ""))
    if hint:
        prompt += f"\nHint:\n{hint}\n"

    prompt += f"\nQuestion:\n{clean_text(row['question'])}\n\n"
    prompt += f"Choices:\n{choice_lines}\n\n"
    prompt += "Answer:"

    return prompt

---
### Model Loading

We will try several versions of LoRA with different hyperparameters to choose the best one.

In [14]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

In [15]:
def load_model():
    processor = AutoProcessor.from_pretrained(MODEL_ID)

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    base_model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )

    model = get_peft_model(base_model, peft_config)
    model.to(device)

    return model, processor

---
### Index-Only Scoring

In [16]:
@torch.no_grad()
def score_indices_for_row(row, prompt_builder, score_mode="sum"):
    image = Image.open(IMAGE_ROOT / row["image_path"]).convert("RGB")
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    answer_texts = [f" {i}" for i in range(num_choices)]
    full_texts = [prompt + ans for ans in answer_texts]

    prompt_inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    prompt_len = prompt_inputs["input_ids"].shape[1]

    full_inputs = processor(
        text=full_texts,
        images=[image] * num_choices,
        return_tensors="pt",
        padding=True,
    )

    full_inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in full_inputs.items()
    }

    input_ids = full_inputs["input_ids"]
    attention_mask = full_inputs.get("attention_mask", torch.ones_like(input_ids))

    outputs = model(**full_inputs)
    logits = outputs.logits.float()

    shift_logits = logits[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention = attention_mask[:, 1:]

    log_probs = F.log_softmax(shift_logits, dim=-1)

    token_log_probs = log_probs.gather(
        dim=-1,
        index=shift_input_ids.unsqueeze(-1),
    ).squeeze(-1)

    answer_mask = torch.zeros_like(shift_input_ids, dtype=torch.bool)

    start = max(prompt_len - 1, 0)
    answer_mask[:, start:] = True

    answer_mask = answer_mask & shift_attention.bool()

    scores = []

    for i in range(num_choices):
        vals = token_log_probs[i][answer_mask[i]]

        if vals.numel() == 0:
            scores.append(-1e9)
        elif score_mode == "sum":
            scores.append(vals.sum().item())
        elif score_mode == "mean":
            scores.append(vals.mean().item())
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

    return scores

In [18]:
# Since we will train the model this time, we need to use the training version of the scoring function.
# Same idea as Model 4 scoring, but WITHOUT torch.no_grad(), so LoRA can learn from the choice scores.
def score_indices_for_row_train(row, prompt_builder, score_mode="sum"):
    prompt = prompt_builder(row)

    num_choices = int(row["num_choices"])
    image_path = IMAGE_ROOT / row["image_path"]
    image = Image.open(image_path).convert("RGB")

    scores = []

    for choice_idx in range(num_choices):
        answer_text = " " + str(choice_idx)
        full_text = prompt + answer_text

        prompt_inputs = processor(
            text=[prompt],
            images=[image],
            return_tensors="pt",
        )

        prompt_len = prompt_inputs["input_ids"].shape[1]

        inputs = processor(
            text=[full_text],
            images=[image],
            return_tensors="pt",
        )

        model_device = next(model.parameters()).device

        inputs = {
            k: v.to(model_device) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

        input_ids = inputs["input_ids"]

        outputs = model(**inputs)
        logits = outputs.logits.float()

        log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)
        target_ids = input_ids[:, 1:]

        token_log_probs = log_probs.gather(
            dim=2,
            index=target_ids.unsqueeze(-1),
        ).squeeze(-1)

        start = max(prompt_len - 1, 0)
        answer_token_log_probs = token_log_probs[:, start:]

        if score_mode == "sum":
            score = answer_token_log_probs.sum()
        elif score_mode == "mean":
            score = answer_token_log_probs.mean()
        else:
            raise ValueError("score_mode must be 'sum' or 'mean'")

        scores.append(score)

    return torch.stack(scores)

In [19]:
def predict_row(row, prompt_builder, score_mode="sum"):
    scores = score_indices_for_row(row, prompt_builder, score_mode=SCORE_MODE)
    pred = int(np.argmax(scores))
    return pred, scores

---
### Training

In [20]:
def train_one_fold(fold_id, train_idx, val_idx):

    print(f"\n===== FOLD {fold_id} =====")

    model, processor = load_model()

    train_subset = train_df.iloc[train_idx]
    val_subset = train_df.iloc[val_idx]

    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

    EPOCHS = 2  # start small

    model.train()

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}")

        for _, row in tqdm(train_subset.iterrows(), total=len(train_subset)):

            image = load_image(IMAGE_ROOT / row["image_path"])
            prompt = build_prompt(row)

            label = f" {row['answer']}"

            inputs = processor(
                text=[prompt + label],
                images=[image],
                return_tensors="pt",
                padding=True,
            )

            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs, labels=inputs["input_ids"])
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    return model, processor, val_subset

In [22]:
def compute_scores(model, processor, df, save_path):

    model.eval()
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df)):

        scores = score_indices_for_row(
            row,
            prompt_builder=build_prompt,
            score_mode="mean",
        )

        results.append({
            "id": row["id"],
            "scores": scores,
            "num_choices": int(row["num_choices"]),
        })

    with open(save_path, "wb") as f:
        pickle.dump(results, f)

    print("Saved:", save_path)

In [23]:
# all_models = []

# for fold_id, (train_idx, val_idx) in enumerate(folds):

#     model, processor, val_subset = train_one_fold(fold_id, train_idx, val_idx)

#     # Save validation scores
#     compute_scores(
#         model,
#         processor,
#         val_subset,
#         f"../runs/fold{fold_id}_val.pkl"
#     )

#     # Save test scores
#     compute_scores(
#         model,
#         processor,
#         test_df,
#         f"../runs/fold{fold_id}_test.pkl"
#     )

#     all_models.append((model, processor))

In [24]:
torch.cuda.empty_cache()

gc.collect()

161

In [25]:
all_models = []

START_FOLD = 4   # Resume training

for fold_id, (train_idx, val_idx) in enumerate(folds):

    if fold_id < START_FOLD:
        continue

    print(f"\n===== FOLD {fold_id} =====")

    model, processor, val_subset = train_one_fold(fold_id, train_idx, val_idx)

    compute_scores(
        model,
        processor,
        val_subset,
        f"../runs/fold{fold_id}_val.pkl"
    )

    compute_scores(
        model,
        processor,
        test_df,
        f"../runs/fold{fold_id}_test.pkl"
    )

    # Free memory after each fold
    del model
    torch.cuda.empty_cache()
    gc.collect()


===== FOLD 4 =====

===== FOLD 4 =====


/home/devvingduo/miniforge3/envs/pixels-to-predictions/lib/python3.10/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Epoch 1


  0%|          | 0/2488 [00:00<?, ?it/s]

Epoch 2


  0%|          | 0/2488 [00:00<?, ?it/s]

  0%|          | 0/621 [00:00<?, ?it/s]

Saved: ../runs/fold4_val.pkl


  0%|          | 0/1008 [00:00<?, ?it/s]

Saved: ../runs/fold4_test.pkl


---
### Evaluation

Ensemble validation accuracy to check quality.

In [29]:
# Collect all fold dictionaries
fold_dicts = []

for f in range(5):
    path = f"../runs/fold{f}_val.pkl"
    
    with open(path, "rb") as f_in:
        data = pickle.load(f_in)
    
    d = {x["id"]: x["scores"] for x in data}
    fold_dicts.append(d)

print("Loaded fold dicts")

# Get all IDs from all folds 
all_ids = set()
for d in fold_dicts:
    all_ids.update(d.keys())

all_ids = list(all_ids)

print("Total unique IDs:", len(all_ids))

# Build predictions using ONLY available folds
final_preds = []
final_ids = []

for id_ in all_ids:
    collected = []

    for f in range(5):
        if id_ in fold_dicts[f]:
            collected.append(fold_dicts[f][id_])

    if len(collected) == 0:
        continue

    scores = np.array(collected)    # (k, num_choices)
    avg_scores = scores.mean(axis=0)    # ensemble

    pred = int(np.argmax(avg_scores))

    final_preds.append(pred)
    final_ids.append(id_)

print("Predictions computed:", len(final_preds))

# Get labels 
labels = train_df.set_index("id").loc[final_ids]["answer"].values

acc = np.mean(np.array(final_preds) == labels)

print("CV Ensemble Accuracy:", acc)

Loaded fold dicts
Total unique IDs: 3109
Predictions computed: 3109
CV Ensemble Accuracy: 0.545513026696687


---
### Submission

Full test ensemble.

In [30]:
# Load all test fold predictions
fold_test_dicts = []

for f in range(5):
    path = f"../runs/fold{f}_test.pkl"
    
    with open(path, "rb") as f_in:
        data = pickle.load(f_in)
    
    d = {x["id"]: x["scores"] for x in data}
    fold_test_dicts.append(d)

print("Loaded test folds")

# All test IDs
ids = list(fold_test_dicts[0].keys())

# Ensemble predictions
final_preds = []

for id_ in ids:
    scores = np.array([fold_test_dicts[f][id_] for f in range(5)])
    avg_scores = scores.mean(axis=0)

    pred = int(np.argmax(avg_scores))
    final_preds.append(pred)

# Create submission 
submission = pd.DataFrame({
    "id": ids,
    "answer": final_preds
})

submission.to_csv("../submission.csv", index=False)

print("Saved submission!")

Loaded test folds
Saved submission!


This achieved a score of 0.54728 on Kaggle.